In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector = embeddings.embed_query("hello, world!")

print(vector)

[-0.0102722915, -0.0023818077, 0.0279904, -0.0067098644, -0.013417502, 0.017604716, -0.005260987, 0.0070480662, -0.00049646996, -0.0383253, -0.0020339957, 0.011152104, -0.0075361487, 0.0050311787, 0.029766537, 0.0068923356, 0.013343806, -0.021058092, -0.01727262, -0.00850923, -0.0059108557, 0.005205297, -0.002630823, 0.03359112, -4.304794e-05, 0.030150875, 0.020684168, -0.003247388, -0.02607466, 0.11794204, 0.00074672606, -0.011136227, -0.0002506568, -0.024933197, 0.025619557, -0.005790998, 0.005279609, -0.0009416169, -0.0017074682, -0.020126559, 0.019258993, -0.0035299452, 0.011687503, 0.006740229, -0.0036968521, -0.0059628184, -0.026179168, -0.013317835, 0.007546382, 0.0029619082, -0.020488648, 0.014980944, 0.00063713366, -0.010201527, -0.016404057, -0.0020851595, -0.019533476, -0.012128767, -0.024494877, 0.01036521, 0.023698105, -0.009647389, 0.013608148, 0.02004486, -0.011885679, -0.0038075408, 0.012490492, -0.0023376686, 0.007772504, 0.0034224237, 0.025826804, 0.017880881, -0.0039

In [ ]:
from anyio import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


path = Path("post.txt")
loader = TextLoader(path, encoding="utf-8")
document = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200,    separators=[" ", ""]
)
texts = text_splitter.split_documents(document)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector = embeddings.embed_documents([text.page_content for text in texts])

print(vector)

[[0.0111759035, 0.032749467, -0.0046028164, 0.013158552, 0.029137526, -0.020019183, -0.02288548, 0.024284702, 0.03904405, -0.04355427, -0.021458253, 0.0018314667, 0.010138611, -0.009779308, -0.00030169825, 0.015824692, 0.027209314, -0.021673232, -0.01535788, -0.021418663, 0.01810593, 0.008894758, 0.01410148, 0.01383782, -0.008491908, 0.0093165655, 0.004720459, -0.0020476033, -0.027471682, 0.10794855, -0.008914822, -0.0032343566, -0.014531359, 0.0067879697, 0.0011328807, -0.024053145, 0.012849324, 0.025687333, 0.0018956164, -0.016159037, 0.012075122, 0.0023877418, 0.005890746, -0.005138889, 0.0046396405, -0.0069805286, 0.02183641, 0.009393651, 0.0074442425, 0.008630575, -0.012967532, -0.0071786027, 0.016658684, 0.006245505, 0.031762213, -0.0029089693, 0.009697838, 0.0066173566, 0.0121159945, 0.00828428, -0.012808299, 0.008588084, 0.029866239, 0.01705642, 0.014019959, -0.021860834, 0.007238384, -0.02167372, -0.0086210985, -0.020867996, -0.018413063, -0.00036782888, -0.029475724, -0.00288

# Vector DB

In [4]:
from pprint import pprint
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings


from anyio import Path

path = Path("movies.json")

import json

with open(path, encoding="utf-8", mode="r") as f:
    movies = json.load(f)

documents=[Document(
    page_content=movie["description"],
    metadata={"title": movie["title"], "genre": ','.join(movie["genre"])}) 
    for movie in movies
    ]


embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

vector_store = Chroma.from_documents(
    documents,
    embedding=embeddings,
    persist_directory="./movies_db"
)



In [6]:
query = vector_store.similarity_search(query='mystery action movie',k=4)
pprint(query)

[Document(id='8dbbf143-9fae-49ff-a5e6-c0c80a31e33f', metadata={'title': 'Knives Out', 'genre': 'Mystery,Comedy,Crime'}, page_content='A famous mystery novelist dies after a family gathering, and a detective investigates the conflicting stories of his relatives and staff. Hidden motives, family tensions, and unexpected clues complicate the search for the truth.'),
 Document(id='0250a9b7-9e63-4487-8ea1-da6ae0f4c530', metadata={'title': 'A Quiet Place', 'genre': 'Horror,Science Fiction,Thriller'}, page_content='A family tries to survive in a world where mysterious creatures hunt by sound. They must communicate in near silence and carefully manage everyday tasks while protecting their children from danger.'),
 Document(id='612d798d-761d-4d65-8d14-e7ae956fa01f', metadata={'genre': 'Comedy,Adventure,Crime', 'title': 'The Grand Budapest Hotel'}, page_content='A legendary hotel concierge and his young lobby assistant become involved in the theft of a valuable painting and a dispute over an inh